# Getting Started with struct-extract-eval

Evaluate how well an LLM extracted structured JSON from text -- in 3 steps:

1. **Provide data** -- gold (ground truth) and extracted (LLM output) as lists of dicts
2. **Create an eval schema** -- infer from gold, then add default comparators
3. **Evaluate** -- get precision, recall, F1, per-field breakdown

## Step 1: Provide Data

Gold is what the correct extraction should look like. Extracted is what the LLM actually produced. Both are lists of dicts -- one dict per record, matched by position (gold[0] vs extracted[0], etc.).

In [1]:
GOLD = [
    {
        "material": "Silicon",
        "temperature": 300.0,
        "method": "Chemical Vapor Deposition",
        "thickness": 50.5,
    },
    {
        "material": "Copper",
        "temperature": 450.0,
        "method": "Sputtering",
        "thickness": 120.0,
    },
    {
        "material": "Gold",
        "temperature": 200.0,
        "method": "Evaporation",
        "thickness": 30.0,
    },
]

EXTRACTED = [
    {
        "material": "Silicon",       # correct
        "temperature": 302.0,        # close but not exact
        "method": "CVD",             # synonym -- won't match with exact comparator
        "thickness": 50.5,           # correct
    },
    {
        "material": "Copper",        # correct
        "temperature": 450.0,        # correct
        "method": "Sputtering",      # correct
        # thickness missing           -> omission
    },
    {
        "material": "Gold",          # correct
        "temperature": 200.0,        # correct
        "method": "Evaporation",     # correct
        "thickness": 30.0,           # correct
        "purity": 0.999,             # extra field -> hallucination
    },
]

## Step 2: Create an Eval Schema

Two calls:
- `infer_schema(GOLD)` -- builds a JSON schema from your gold data (types, required/optional fields)
- `annotate_xeval(schema)` -- adds default `x-eval-compare` to each leaf field (strings -> `exact`, numbers -> `numeric`)

In [2]:
import json
from struct_extract_eval import infer_schema, annotate_xeval

eval_schema = infer_schema(GOLD)
annotate_xeval(eval_schema)

print(json.dumps(eval_schema, indent=2))

{
  "type": "object",
  "properties": {
    "material": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "method": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "temperature": {
      "type": "number",
      "x-eval-compare": "numeric"
    },
    "thickness": {
      "type": "number",
      "x-eval-compare": "numeric"
    }
  }
}


## Step 3: Evaluate

In [3]:
from struct_extract_eval import evaluate

run = evaluate(GOLD, EXTRACTED, schema=eval_schema)

print(f"Records:        {run.total_records}")
print(f"Fields scored:  {run.total_fields}")
print(f"Precision:      {run.mean_precision:.3f}")
print(f"Recall:         {run.mean_recall:.3f}")
print(f"F1:             {run.mean_f1:.3f}")
print(f"Omissions:      {run.total_omissions}")
print(f"Hallucinations: {run.total_hallucinations}")

Records:        3
Fields scored:  13
Precision:      0.767
Recall:         0.750
F1:             0.749
Omissions:      1
Hallucinations: 1


## Inspect: Per-Field Breakdown

Which fields does the extractor get right? Which does it struggle with?

In [4]:
print(f"{'Field':<20} {'Score':>6} {'Match':>6} {'Mis':>6} {'Omit':>6} {'Hall':>6}")
print("-" * 56)
for path, agg in sorted(run.per_field.items()):
    print(f"{path:<20} {agg.mean_score:>6.2f} {agg.matches:>6} {agg.mismatches:>6} "
          f"{agg.omissions:>6} {agg.hallucinations:>6}")

Field                 Score  Match    Mis   Omit   Hall
--------------------------------------------------------
material               1.00      3      0      0      0
method                 0.67      2      1      0      0
purity                 0.00      0      0      0      1
temperature            0.67      2      1      0      0
thickness              0.67      2      0      1      0


## Inspect: Per-Record Detail

Drill into each record to see exactly what matched and what didn't.

In [5]:
for record in run.records:
    print(f"\n--- Record {record.record_id}  (F1: {record.f1:.3f}  P: {record.precision:.3f}  R: {record.recall:.3f}) ---")
    print(f"  {'Field':<15} {'Gold':<30} {'Extracted':<30} {'Score':>5}  {'Status'}")
    for fr in record.field_results:
        g = str(fr.gold_value)[:28]
        e = str(fr.extracted_value)[:28]
        print(f"  {fr.path:<15} {g:<30} {e:<30} {fr.score:>5.1f}  {fr.status}")


--- Record 0  (F1: 0.500  P: 0.500  R: 0.500) ---
  Field           Gold                           Extracted                      Score  Status
  material        Silicon                        Silicon                          1.0  match
  method          Chemical Vapor Deposition      CVD                              0.0  mismatch
  temperature     300.0                          302.0                            0.0  mismatch
  thickness       50.5                           50.5                             1.0  match

--- Record 1  (F1: 0.857  P: 1.000  R: 0.750) ---
  Field           Gold                           Extracted                      Score  Status
  material        Copper                         Copper                           1.0  match
  method          Sputtering                     Sputtering                       1.0  match
  temperature     450.0                          450.0                            1.0  match
  thickness       120.0                          None

## Next: Manual Tweaks to Eval Schema


Things you might notice and want to fix by editing the eval schema:

- **"CVD" vs "Chemical Vapor Deposition"** scored as mismatch -- the `exact` comparator does literal string matching. Use `oneof` with synonyms for fuzzy matching. (For LLM-based semantic comparison, see `demo.ipynb` -- it requires registering a batch comparator first.)
- **temperature 302 vs 300** scored as mismatch -- the default `numeric` comparator uses zero tolerance. Add `{"numeric": {"tolerance": {"rel": 0.01}}}` for 1% relative tolerance.
- **"purity" hallucination** -- the extractor invented a field not in the schema. This is detected as a hallucination and lowers precision.
- **"thickness" omission** (record 1) -- the field is in the schema and in gold, but the extractor didn't produce it. This is detected as an omission and lowers recall.